In [1]:
import os
from pathlib import Path

ROOT_DIR_BRAINTREEBANK = Path(r"C:\Users\simon\PyCharmMiscProject\neuroprobe-dev\braintreebank")
os.environ["ROOT_DIR_BRAINTREEBANK"] = str(ROOT_DIR_BRAINTREEBANK)

assert ROOT_DIR_BRAINTREEBANK.exists(), ROOT_DIR_BRAINTREEBANK

In [2]:
CHECKPOINT_DIR = Path("notebooks/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / "leaderboard_week3_best.pt"
print("Checkpoint will be saved to:", CHECKPOINT_PATH)

Checkpoint will be saved to: notebooks\models\leaderboard_week3_best.pt


In [3]:
# Standard library
import copy
import inspect
import json
import random

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# Scikit-learn
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Neuroprobe
import neuroprobe
import neuroprobe.train_test_splits as neuroprobe_train_test_splits
from neuroprobe import (
    BrainTreebankSubject,
    BrainTreebankSubjectTrialBenchmarkDataset,
    generate_splits_cross_session,
    generate_splits_cross_subject,
    generate_splits_within_session,
)

In [4]:
# =========================
# Repro / device
# =========================
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Benchmark / split
# =========================
TASKS = ["delta_volume", "speech", "pitch", "gpt2_surprisal", "word_gap"]

TEST_SUBJECT_ID = 1
TEST_TRIAL_ID = 2
SUBJECT_IDS = list(range(1, 11))

COORDINATE_SYSTEM = "cortical"   # lock this for all Phase 1/2 runs
USE_VAL_AS_TEST = False

# =========================
# Training
# =========================
BATCH_SIZE = 8
BASE_LR = 2e-4
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 20
PATIENCE = 5
NUM_WORKERS = 0

DROPOUT = 0.1
ATTN_DROPOUT = 0.1

USE_GLOBAL_TRAIN_NORM = False

# =========================
# Signal preprocessing
# =========================
STFT_N_FFT = 64
STFT_HOP = 16
STFT_WIN_LEN = 32
LAPLACIAN_K = 4

# =========================
# Encoder dimensions
# =========================
COORD_DIM = 3
TRUNK_OUT_DIM = 96            # current CNN output width
COORD_EMB_DIM = 32            # slightly richer than 16 for geometry
ELEC_HIDDEN_DIM = 128         # trunk projection output
MODEL_DIM = 128

SUBJ_EMB_DIM = 16
TASK_EMB_DIM = 8

# =========================
# Virtual sensor harmonizer
# =========================
NUM_VIRTUAL_SENSORS = 16      # sweep: 16 / 32 / 64 after first stable run
NUM_SENSOR_HEADS = 4
USE_COORDS_IN_KEYS = True
USE_COORDS_IN_VALUES = False

# =========================
# Sensor refinement
# =========================
USE_SENSOR_SELF_ATTN = True

# Keep this at 1 for now unless I actually implement a stack
NUM_SENSOR_SELF_ATTN_LAYERS = 1

# =========================
# Conditioning choices
# =========================
USE_SUBJECT_EMBEDDING = True
USE_TASK_EMBEDDING = True

# =========================
# Safety / debugging
# =========================
ASSERT_NONEMPTY_ELECTRODE_SET = True
RUN_PERMUTATION_TEST = True
PRINT_REAL_BATCH_CONTRACT = True

print("DEVICE:", DEVICE)
print("COORDINATE_SYSTEM:", COORDINATE_SYSTEM)

DEVICE: cpu
COORDINATE_SYSTEM: cortical


In [5]:
# =========================
# Virtual sensor harmonizer
# =========================
class VirtualSensorHarmonizer(nn.Module):
    def __init__(
            self,
            elec_hidden_dim,
            coord_dim,
            coord_emb_dim,
            model_dim,
            num_virtual_sensors,
            num_heads,
            attn_dropout=0.1,
            dropout=0.1,
            use_coords_in_keys=True,
            use_coords_in_values=False,
            use_sensor_self_attn=True,
    ):
        super().__init__()

        self.use_coords_in_keys = use_coords_in_keys
        self.use_coords_in_values = use_coords_in_values
        self.use_sensor_self_attn = use_sensor_self_attn

        self.coord_mlp = nn.Sequential(
            nn.Linear(coord_dim, coord_emb_dim),
            nn.LayerNorm(coord_emb_dim),
            nn.ReLU(),
            nn.Linear(coord_emb_dim, coord_emb_dim),
        )

        key_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_keys else 0)
        val_in_dim = elec_hidden_dim + (coord_emb_dim if use_coords_in_values else 0)

        self.key_proj = nn.Sequential(
            nn.Linear(key_in_dim, model_dim),
            nn.LayerNorm(model_dim),
        )
        self.val_proj = nn.Sequential(
            nn.Linear(val_in_dim, model_dim),
            nn.LayerNorm(model_dim),
        )

        self.virtual_queries = nn.Parameter(
            torch.randn(num_virtual_sensors, model_dim) * 0.02
        )

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,
        )
        self.cross_attn_norm = nn.LayerNorm(model_dim)
        self.cross_attn_dropout = nn.Dropout(dropout)

        if use_sensor_self_attn:
            self.sensor_self_attn = nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=num_heads,
                dropout=attn_dropout,
                batch_first=True,
            )
            self.sensor_attn_norm = nn.LayerNorm(model_dim)
            self.sensor_attn_dropout = nn.Dropout(dropout)

            self.sensor_ff_norm = nn.LayerNorm(model_dim)
            self.sensor_ff = nn.Sequential(
                nn.Linear(model_dim, model_dim * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(model_dim * 4, model_dim),
            )
            self.sensor_ff_dropout = nn.Dropout(dropout)

        self.out_norm = nn.LayerNorm(model_dim)

    def forward(self, elec_feat, coords, elec_mask):
        assert elec_feat.ndim == 3
        assert coords.ndim == 3
        assert elec_mask.ndim == 2
        assert elec_feat.shape[:2] == coords.shape[:2] == elec_mask.shape
        assert elec_mask.any(dim=1).all(), "Each sample must have at least one real electrode."

        coord_feat = self.coord_mlp(coords)

        key_in = torch.cat([elec_feat, coord_feat], dim=-1) if self.use_coords_in_keys else elec_feat
        val_in = torch.cat([elec_feat, coord_feat], dim=-1) if self.use_coords_in_values else elec_feat

        keys = self.key_proj(key_in)
        values = self.val_proj(val_in)

        B = elec_feat.size(0)
        queries = self.virtual_queries.unsqueeze(0).expand(B, -1, -1)
        key_padding_mask = ~elec_mask.bool()

        query_normed = self.cross_attn_norm(queries)
        attn_out, _ = self.cross_attn(
            query=query_normed,
            key=keys,
            value=values,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        sensors = queries + self.cross_attn_dropout(attn_out)

        if self.use_sensor_self_attn:
            sensor_normed = self.sensor_attn_norm(sensors)
            attn_out, _ = self.sensor_self_attn(
                query=sensor_normed,
                key=sensor_normed,
                value=sensor_normed,
                need_weights=False,
            )
            sensors = sensors + self.sensor_attn_dropout(attn_out)

            ff_out = self.sensor_ff(self.sensor_ff_norm(sensors))
            sensors = sensors + self.sensor_ff_dropout(ff_out)

        sensors = self.out_norm(sensors)
        return sensors

In [6]:
# =========================
# Harmonizer tests
# =========================
harmonizer = VirtualSensorHarmonizer(
    elec_hidden_dim=ELEC_HIDDEN_DIM,
    coord_dim=COORD_DIM,
    coord_emb_dim=COORD_EMB_DIM,
    model_dim=MODEL_DIM,
    num_virtual_sensors=NUM_VIRTUAL_SENSORS,
    num_heads=NUM_SENSOR_HEADS,
    attn_dropout=ATTN_DROPOUT,
    dropout=DROPOUT,
    use_coords_in_keys=USE_COORDS_IN_KEYS,
    use_coords_in_values=USE_COORDS_IN_VALUES,
    use_sensor_self_attn=USE_SENSOR_SELF_ATTN,
).to(DEVICE)

harmonizer.eval()

B, E = 4, 11
elec_feat = torch.randn(B, E, ELEC_HIDDEN_DIM, device=DEVICE)
coords = torch.randn(B, E, COORD_DIM, device=DEVICE)
elec_mask = torch.ones(B, E, dtype=torch.bool, device=DEVICE)

# padded tails
elec_mask[1, -3:] = False
elec_mask[2, -5:] = False

if ASSERT_NONEMPTY_ELECTRODE_SET:
    assert elec_mask.any(dim=1).all(), "Found a sample with all electrodes masked"

with torch.no_grad():
    sensor_tokens = harmonizer(elec_feat, coords, elec_mask)

print("elec_feat      :", elec_feat.shape)
print("coords         :", coords.shape)
print("elec_mask      :", elec_mask.shape, elec_mask.dtype)
print("valid counts   :", elec_mask.sum(dim=1).tolist())
print("sensor_tokens  :", sensor_tokens.shape)

assert sensor_tokens.shape == (B, NUM_VIRTUAL_SENSORS, MODEL_DIM)
assert torch.isfinite(sensor_tokens).all(), "NaN/Inf found in sensor_tokens"

print("shape + finiteness test passed")

# -------------------------
# Permutation invariance test
# -------------------------
if RUN_PERMUTATION_TEST:
    perm = torch.randperm(E, device=DEVICE)

    elec_feat_perm = elec_feat[:, perm, :]
    coords_perm = coords[:, perm, :]
    elec_mask_perm = elec_mask[:, perm]

    with torch.no_grad():
        sensor_tokens_perm = harmonizer(elec_feat_perm, coords_perm, elec_mask_perm)

    max_diff = (sensor_tokens - sensor_tokens_perm).abs().max().item()
    print("permutation max diff:", max_diff)

    assert torch.allclose(sensor_tokens, sensor_tokens_perm, atol=1e-5, rtol=1e-4), \
        f"Permutation invariance failed: max diff = {max_diff}"

    print("permutation invariance test passed")

# -------------------------
# Mask sensitivity test
# -------------------------
mask_drop = elec_mask.clone()
valid_idx = torch.where(mask_drop[0])[0]
mask_drop[0, valid_idx[-2:]] = False

with torch.no_grad():
    sensor_tokens_mask_drop = harmonizer(elec_feat, coords, mask_drop)

mask_diff = (sensor_tokens - sensor_tokens_mask_drop).abs().max().item()
print("mask change max diff:", mask_diff)

assert mask_diff > 0, "Mask sensitivity test failed: output did not change after masking valid electrodes."
print("mask sensitivity test passed")

# -------------------------
# Failure-case check
# -------------------------
bad_mask = elec_mask.clone()
bad_mask[0, :] = False

print("all-masked row present:", (~bad_mask.any(dim=1)).any().item())

try:
    with torch.no_grad():
        _ = harmonizer(elec_feat, coords, bad_mask)
    raise AssertionError("Expected all-masked batch assertion, but forward passed.")
except AssertionError as e:
    print("caught expected assertion:", e)

print("all-masked guard test passed")

elec_feat      : torch.Size([4, 11, 128])
coords         : torch.Size([4, 11, 3])
elec_mask      : torch.Size([4, 11]) torch.bool
valid counts   : [11, 8, 6, 11]
sensor_tokens  : torch.Size([4, 16, 128])
shape + finiteness test passed
permutation max diff: 1.9073486328125e-06
permutation invariance test passed
mask change max diff: 1.3099204301834106
mask sensitivity test passed
all-masked row present: True
caught expected assertion: Each sample must have at least one real electrode.
all-masked guard test passed


Great! So all tests passed:
- Shape is correct
- Permutation invariance passed with tiny max diff
- Mask sensity is great
- All-masked guard works as intended

I'll run some tests to see if the real dataset creates any issues